In [9]:
import xarray as xr
from pathlib import Path


from era5vis.helpers import add_wind_speed_dir

In [12]:
import xarray as xr
from pathlib import Path


from era5vis.helpers import add_wind_speed_dir

ds = xr.load_dataset('era5vis-main/data/model_climate/data_stream-moda_stepType-avgua.nc')

ds = add_wind_speed_dir(ds, u_name="u", v_name="v")

# Group by year and month, then take the mean for each month across all years (i.e., the mean annual cycle)
yearly_mean_cycle = ds.groupby('valid_time.month').mean(dim='valid_time')


out = Path("era5vis-main/data/model_climate/model_clim.nc")
out.parent.mkdir(parents=True, exist_ok=True)  # make sure folder exists
yearly_mean_cycle.to_netcdf(out)

In [10]:
import xarray as xr
from pathlib import Path

from era5vis.helpers import add_wind_speed_dir

ds_api = xr.load_dataset(
    "era5vis-main/data/tmp/data_stream-moda_stepType-avgua-3.nc"
)

# add wind speed & direction
ds_api = add_wind_speed_dir(ds_api, u_name="u", v_name="v")

# # extract month number (1–12)
# month_value = int(ds_api["valid_time"].dt.month.values[0])

# # remove valid_time dimension
# ds_api = ds_api.isel(valid_time=0, drop=True)

# # add month dimension indexed with the correct month number
# ds_api = ds_api.expand_dims(month=[month_value])

# # optional: make month an int coordinate (clean)
# ds_api["month"] = ds_api["month"].astype("int64")

ds_api.drop_dims("valid_time")

# write output
out_api = Path("era5vis-main/data/tmp/test_tmp_with_wind.nc")
out_api.parent.mkdir(parents=True, exist_ok=True)
ds_api.to_netcdf(out_api)

# Make Model Topo

In [8]:
topo_ds = xr.load_dataset('/Users/jakobwerkgarner/code/Scipro/project/era5vis-main/data/model_terrain/4038074f891f7b15138bb9d8fd46112.nc')



g = 9.81  # m/s^2

# convert geopotential to geometric height (m)
elevation_m = topo_ds["z"] / g
elevation_m.name = "elevation"
elevation_m.attrs.update({"long_name": "Surface elevation", "units": "m"})

# convert height (m) -> approximate pressure (hPa) using standard atmosphere
def height_to_pressure_hpa(z_m: xr.DataArray) -> xr.DataArray:
    T0 = 288.15
    L = 0.0065
    g0 = 9.80665
    M = 0.0289644
    R = 8.3144598
    p0 = 100000.0  # Pa

    z = xr.where(z_m < 0.0, 0.0, z_m)
    exponent = (g0 * M) / (R * L)
    p_pa = p0 * (1.0 - (L * z / T0)) ** exponent
    return p_pa / 100.0  # hPa


p_sfc_hpa = height_to_pressure_hpa(elevation_m)
p_sfc_hpa.name = "p_sfc"
p_sfc_hpa.attrs.update(
    {"long_name": "Surface pressure (standard atmosphere from elevation)", "units": "hPa"}
)

# save both elevation and pressure-topography
out_ds = xr.Dataset({"elevation": elevation_m, "p_sfc": p_sfc_hpa})
out_ds.to_netcdf("era5vis-main/era5vis/data/model_topo_pressure.nc")


In [ ]:
ds_api

<xarray.Dataset> Size: 83MB
Dimensions:         (valid_time: 1, pressure_level: 13, latitude: 221,
                     longitude: 401)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 8B 2015-09-01
  * pressure_level  (pressure_level) float64 104B 1e+03 950.0 ... 200.0 100.0
  * latitude        (latitude) float64 2kB 90.0 89.75 89.5 ... 35.5 35.25 35.0
  * longitude       (longitude) float64 3kB -55.0 -54.75 -54.5 ... 44.75 45.0
    number          int64 8B 0
    expver          <U4 16B '0001'
Data variables: (12/18)
    d               (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    cc              (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    z               (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    o3              (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    pv              (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    r               (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    ...              ...
    u               (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    v               (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    w               (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    vo              (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    wspd            (valid_time, pressure_level, latitude, longitude) float32 5MB ...
    wdir            (valid_time, pressure_level, latitude, longitude) float32 5MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-10T13:41 GRIB to CDM+CF via cfgrib-0.9.1...

In [14]:
AREA = [90, -55, 35, 35]  # [North, West, South, East]

df = xr.open_dataset("era5vis-main/era5vis/data/model_clim.nc")

In [16]:
print(df)

<xarray.Dataset> Size: 442MB
Dimensions:         (month: 12, pressure_level: 13, latitude: 221,
                     longitude: 401)
Coordinates:
  * month           (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * pressure_level  (pressure_level) float64 104B 1e+03 950.0 ... 200.0 100.0
  * latitude        (latitude) float64 2kB 90.0 89.75 89.5 ... 35.5 35.25 35.0
  * longitude       (longitude) float64 3kB -55.0 -54.75 -54.5 ... 44.75 45.0
    number          int64 8B ...
Data variables:
    z               (month, pressure_level, latitude, longitude) float32 55MB ...
    q               (month, pressure_level, latitude, longitude) float32 55MB ...
    crwc            (month, pressure_level, latitude, longitude) float32 55MB ...
    t               (month, pressure_level, latitude, longitude) float32 55MB ...
    u               (month, pressure_level, latitude, longitude) float32 55MB ...
    v               (month, pressure_level, latitude, longitude) float32 55MB ...
    wspd     